In [7]:
from pathlib import Path
from PIL import Image

root = Path("../../data/ADNI/AD_NC")
files = {(s, c): sorted((root/s/c).glob("*")) for s in ("train", "test") for c in ("AD", "NC")}
for k, v in files.items():
    print(k, len(v))
for f in files[("train", "AD")]:
# f = files[("train", "AD")][]
    print(Image.open(f).size, Image.open(f).mode)
    # print([x.name for x in files[("train", "AD")]])  # check how patient ID appears

# patient overlap between train and test (assumes names like 388206_78.jpeg)
pid = lambda f: f.name.split("_")[0]
train_ids = {pid(f) for (s, _), v in files.items() if s == "train" for f in v}
test_ids = {pid(f) for (s, _), v in files.items() if s == "test" for f in v}
print(len(train_ids), len(test_ids), "overlap:", len(train_ids & test_ids))

('train', 'AD') 10400
('train', 'NC') 11120
('test', 'AD') 4460
('test', 'NC') 4540
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 240) L
(256, 

KeyboardInterrupt: 

In [4]:
import torch, torchvision, torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
model = torchvision.models.convnext_tiny(weights=None)
model.classifier[2] = nn.Linear(768, 2)
model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
print("params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

for bs in [8, 16, 32, 64, 128]:
    try:
        if device == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        x = torch.randn(bs, 3, 224, 224, device=device)
        y = torch.randint(0, 2, (bs,), device=device)
        for _ in range(3):
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
        vram = f"{torch.cuda.max_memory_allocated() / 2**20:.0f} MiB" if device == "cuda" else "n/a (CPU)"
        print(f"bs={bs}: loss={loss.item():.3f}, peak VRAM={vram}")
    except torch.cuda.OutOfMemoryError:
        print(f"bs={bs}: out of memory")
        break

params: 27.821666 M
bs=8: loss=0.019, peak VRAM=1321 MiB
bs=16: loss=0.181, peak VRAM=2119 MiB
bs=32: loss=0.488, peak VRAM=3869 MiB
bs=64: loss=0.683, peak VRAM=7200 MiB
bs=128: loss=0.717, peak VRAM=13925 MiB


In [1]:
import torch
print(torch.__version__, torch.cuda.is_available())

[WARNING] offload-arch failed with return code 1
[stderr] 
2.9.1+rocm7.2.1 True
